# CatBoost2 Cloud — Predicción Local + Kaggle Submission

**Corre localmente** después de descargar el modelo entrenado en Kaggle Notebooks.

**Archivos necesarios en la misma carpeta:**
- `catboost2_cloud_best.cbm` — modelo entrenado (CatBoost2: 5-fold CV + cat features)
- `catboost2_cloud_metadata.json` — metadata

**Dataset necesario:**
- `data/processed/features_test_cat.parquet` — test con 30 features numéricas + 6 categóricas

In [ ]:
from catboost import CatBoostClassifier
import pandas as pd
import numpy as np
import json
import time
import warnings
warnings.filterwarnings('ignore')
from pathlib import Path

import catboost
print(f'CatBoost version: {catboost.__version__}')
print('Imports OK')

In [ ]:
MODEL_DIR = Path('.')
DATA_DIR  = Path('../../data/processed')

model_file = MODEL_DIR / 'catboost2_cloud_best.cbm'
meta_file  = MODEL_DIR / 'catboost2_cloud_metadata.json'

print(f'MODEL_DIR : {MODEL_DIR.resolve()}')
for f in [model_file, meta_file]:
    status = '✓ encontrado' if f.exists() else '✗ NO encontrado — descargar desde Kaggle'
    print(f'  {f.name} : {status}')

In [ ]:
with open(meta_file, 'r') as f:
    meta = json.load(f)

print('Metadata del modelo:')
print(f'  Train AUC         : {meta["train_auc"]}')
print(f'  CV AUC (5-fold)   : {meta["best_cv_auc"]}')
print(f'  n_rounds          : {meta["best_n_rounds"]}')
print(f'  n_features total  : {len(meta["feature_cols"])}')
print(f'  cat_features      : {meta["cat_cols"]}')
print(f'  Método evaluación : {meta["method"]}  ({meta["n_folds"]}-fold)')
print(f'  GPU usado         : {meta["use_gpu"]}')
print(f'  CatBoost ver      : {meta["catboost_version"]}')
print(f'  Timestamp         : {meta["timestamp"]}')
print('  Hiperparámetros:')
for k, v in meta['best_params'].items():
    print(f'    {k:<24}: {v}')

In [ ]:
print(f'Cargando modelo: {model_file.name}  ({model_file.stat().st_size/1e6:.2f} MB)...')
model = CatBoostClassifier()
model.load_model(str(model_file))
print('Modelo cargado ✓')
print(f'  Iteraciones : {model.tree_count_}')

In [ ]:
# Lee features_test_cat.parquet (incluye las 6 columnas categóricas para CatBoost)
print('Cargando features_test_cat.parquet...')
df_test      = pd.read_parquet(DATA_DIR / 'features_test_cat.parquet')
sk_ids_test  = df_test['SK_ID_CURR'].values
feature_cols = meta['feature_cols']
cat_cols     = meta['cat_cols']

# Las columnas categóricas se pasan a CatBoost como strings (nativamente)
# No se encodean manualmente — CatBoost aplica su target encoding internamente
cat_features_idx = [feature_cols.index(c) for c in cat_cols]

X_test = df_test[feature_cols]   # DataFrame, preserva dtypes

print(f'  Test shape      : {X_test.shape}')
print(f'  Features totales: {len(feature_cols)}')
print(f'  Cat features    : {cat_cols}')
print(f'  NaNs en num.    : {X_test[[c for c in feature_cols if c not in cat_cols]].isnull().sum().sum():,}')

In [ ]:
print('Generando predicciones...')
y_test_prob = model.predict_proba(X_test)[:, 1]

print(f'  Predicciones generadas : {len(y_test_prob):,}')
print(f'  Score mínimo           : {y_test_prob.min():.5f}')
print(f'  Score máximo           : {y_test_prob.max():.5f}')
print(f'  Score medio            : {y_test_prob.mean():.5f}')
print(f'  % predicho como default: {(y_test_prob > 0.5).mean()*100:.2f}%')

In [ ]:
submission = pd.DataFrame({'SK_ID_CURR': sk_ids_test, 'TARGET': y_test_prob})
out_path   = DATA_DIR / 'submission_cloud_catboost2.csv'
submission.to_csv(out_path, index=False)

print(f'Submission guardado: {out_path}')
print(f'Shape: {submission.shape}')
display(submission.head())

## Kaggle Submission — AUC Test (Public Leaderboard)

> **Límite**: 5 submissions/día.

In [ ]:
from kaggle import KaggleApi

COMPETITION = 'home-credit-default-risk'
_msg = (f"CatBoost2 Cloud | "
        f"CV AUC (5-fold)={meta['best_cv_auc']:.5f} | "
        f"train AUC={meta['train_auc']:.5f} | "
        f"rounds={meta['best_n_rounds']} | "
        f"cat_features={len(meta['cat_cols'])}")

def _as_str(x):
    return '' if x is None else str(x)

def _get_score(api, comp, file_name, message, poll=10, timeout=300):
    start = time.time()
    while time.time() - start < timeout:
        subs = api.competition_submissions(comp)
        matched = next(
            (s for s in subs
             if _as_str(getattr(s, 'file_name', None)) == file_name
             and _as_str(getattr(s, 'description', None)) == message),
            subs[0] if subs else None
        )
        if matched:
            pub    = _as_str(getattr(matched, 'public_score',  None))
            status = _as_str(getattr(matched, 'status',        None))
            elapsed = int(time.time() - start)
            print(f'  [{elapsed:>3}s] status={status!r}  public_score={pub!r}')
            if pub and pub.lower() not in ('', 'none', 'null', '-'):
                priv = _as_str(getattr(matched, 'private_score', None))
                return float(pub), (float(priv) if priv and priv.lower() not in ('','none','null','-') else None)
        time.sleep(poll)
    return None, None

_api = KaggleApi()
_api.authenticate()

print(f'Enviando: {_msg}')
_api.competition_submit(file_name=str(out_path), message=_msg, competition=COMPETITION)
print('Esperando scoring (poll 10 s / máx 5 min)...')

public_auc, private_auc = _get_score(_api, COMPETITION, out_path.name, _msg)

print(f'\n' + '=' * 65)
print(f'RESULTADO KAGGLE — CATBOOST2 CLOUD')
print(f'=' * 65)
print(f'  AUC test Public  LB    : {public_auc}')
print(f'  AUC test Private LB    : {private_auc}')
print(f'  CV AUC (5-fold)        : {meta["best_cv_auc"]:.5f}')
if public_auc:
    print(f'  Gap CV - Public LB     : {abs(meta["best_cv_auc"] - public_auc):.5f}')
print(f'=' * 65)